# 09 · scANVI labels, Harmony and scib metrics

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`cluster_data.assign_remaining_cells`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cluster_data/cluster_data.wdl) + [`integrate_harmony`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) + [`artifact_metrics`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`annotate`](https://github.com/ASAP-CRN/sc-tools/blob/main/src/sc_tools/annotate.py) → `sc-annotate` (sc-tools only; replaces `label_scanvi` and `add_harmony`) |

`sc-annotate` runs the workflow's last three modelling steps on one object:

1. **Re-attach MapMyCells labels.** It reads the results CSV, lower-cases the taxonomy level names (newer references capitalize them), applies the workflow's `summarize_mmc_results` (confident calls only, as in notebook 06), and replaces any MMC columns in `obs`. Cells with no MMC result are `Unknown`. Because the labels come in at this step, you can label with a **different reference** (re-run notebook 04 with another taxonomy) without retraining scVI.
2. **scANVI.** `SCANVI.from_scvi_model(labels_key="cell_type", unlabeled_category="Unknown")` starts from the trained scVI model and adds a classifier. It is trained semi-supervised on the confident labels and then predicts a label for **every** cell (`obs['C_scANVI']`). It also writes a label-aware latent space (`obsm['X_scANVI']`). Before training, the inputs are checked for problems that make scANVI diverge (non-finite or negative counts, NaN in `X_scVI`, missing labels). Training settings follow `integrate_scvi` (epoch heuristic, at least 10; gradient clipping).
3. **Harmony.** It corrects the PCA embedding (`X_pca`) for `batch_id` on the GPU (`rapids_singlecell`), adding `obsm['X_pca_harmony']`. This is a second, independent integration kept for comparison; it does not use scVI.
4. **scib metrics.** It benchmarks `X_pca` (unintegrated), `X_scVI` and `X_pca_harmony` with `C_scANVI` as the labels (see notebook 10).

**Requires a GPU.**

**Inputs**
- `${cohort_id}.umap_clustered.h5ad` (with `X_scVI`, `X_pca`)
- `${cohort_id}_scvi_model/`
- `${cohort_id}.mmc_otf_mapping.${mmc_taxonomy}.results.csv`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.final.h5ad`
- `${cohort_id}_scanvi_model/` (and `.scanvi_model.tar.gz`)
- `${cohort_id}.scanvi_cell_types.parquet`
- `scib_report_dir/scib_report.csv`, `scib_results.svg`
- `${cohort_id}.final_metadata.csv`

**Previous:** [08_clustering_umap](08_clustering_umap.ipynb) · **Next:** [10_artifact_metrics](10_artifact_metrics.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Run `sc-annotate`

In [ ]:
mmc_csv = cfg.out(f"mmc_otf_mapping.{cfg.MMC_TAXONOMY}.results.csv")
scanvi_dir = cfg.RUN_DIR / f"{cfg.COHORT_ID}_scanvi_model"
cfg.run(
    "sc-annotate",
    "--workflow-name", cfg.WORKFLOW_NAME,
    "--batch-key", cfg.BATCH_KEY,
    "--latent-key", cfg.SCANVI_LATENT_KEY,
    "--predictions-key", cfg.SCANVI_PREDICTIONS_KEY,
    "--adata-input", cfg.out("umap_clustered.h5ad"),
    "--scvi-outputs-dir", cfg.RUN_DIR / f"{cfg.COHORT_ID}_scvi_model",
    "--mmc-results", mmc_csv,
    "--adata-output", cfg.out("final.h5ad"),
    "--output-scanvi-dir", scanvi_dir,
    "--output-cell-types-file", cfg.out("scanvi_cell_types.parquet"),
    "--output-report-dir", "scib_report_dir",
)

Archive the scANVI model, rename the scib outputs, and write `final_metadata.csv` with the UMAP coordinates, as the WDL tasks do.

In [ ]:
import tarfile

with tarfile.open(cfg.out("scanvi_model.tar.gz"), "w:gz") as tar:
    tar.add(scanvi_dir, arcname=scanvi_dir.name)

report_dir = cfg.RUN_DIR / "scib_report_dir"
for name in ["scib_report.csv", "scib_results.svg"]:
    if (report_dir / name).exists():
        (report_dir / name).replace(report_dir / f"{cfg.COHORT_ID}.{name}")

adata = sc.read_h5ad(cfg.out("final.h5ad"))
metadata = adata.obs.copy()
metadata["UMAP_1"] = adata.obsm["X_umap"][:, 0]
metadata["UMAP_2"] = adata.obsm["X_umap"][:, 1]
metadata.to_csv(cfg.out("final_metadata.csv"), index=True)
adata

## 2. MapMyCells vs. scANVI

Among confidently mapped cells, scANVI should mostly agree with MapMyCells; the `Unknown` row shows how the remaining cells were assigned.

In [ ]:
agree = (adata.obs["cell_type"] == adata.obs[cfg.SCANVI_PREDICTIONS_KEY])
labelled = adata.obs["cell_type"] != "Unknown"
print(f"scANVI agrees with MapMyCells on {100 * agree[labelled].mean():.1f}% of confidently mapped cells")
pd.crosstab(adata.obs["cell_type"], adata.obs[cfg.SCANVI_PREDICTIONS_KEY])

In [ ]:
sc.pl.umap(adata, color=["cell_type", cfg.SCANVI_PREDICTIONS_KEY], ncols=1, frameon=False)

## 3. scVI vs. Harmony embeddings

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, basis in zip(axes, ["X_pca", "X_pca_harmony"]):
    sc.pl.embedding(adata, basis=basis, color="batch_id", ax=ax, show=False, frameon=False, title=basis)
plt.show()

## 4. Compare with the curated release

In [ ]:
released_csv = cfg.curated("final_metadata.csv")
if released_csv.exists():
    released = pd.read_csv(released_csv, index_col=0, usecols=lambda c: c in {"Unnamed: 0", "cell_type", "C_scANVI"})
    shared = adata.obs_names.intersection(released.index)
    print(f"cells: this run {adata.n_obs:,}, release {len(released):,}, shared {len(shared):,}")
    print(f"same C_scANVI label: {100 * (adata.obs.loc[shared, 'C_scANVI'].astype(str) == released.loc[shared, 'C_scANVI'].astype(str)).mean():.1f}%")
    display(pd.crosstab(released.loc[shared, "C_scANVI"], adata.obs.loc[shared, "C_scANVI"]).rename_axis(index="release", columns="this run"))
else:
    print(f"{released_csv.name} not in the curated release")

**Previous:** [08_clustering_umap](08_clustering_umap.ipynb) · **Next:** [10_artifact_metrics](10_artifact_metrics.ipynb)